# Notebook 09 — Phase 2: Fraud Prediction API and Tests



In [1]:
# Cell 1 — Install API and test dependencies in the active notebook environment
%pip install fastapi uvicorn httpx pytest joblib scikit-learn pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — Set project paths and load the saved Phase 1 pipeline
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

# This notebook should be saved inside the project's notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "models").exists() and (PROJECT_ROOT.parent / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = PROJECT_ROOT / "models" / "fraud_detection_pipeline.pkl"
METADATA_PATH = PROJECT_ROOT / "models" / "fraud_detection_metadata.json"

print("Project root:", PROJECT_ROOT)
print("Model exists:", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

if not MODEL_PATH.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find the saved model artifacts. Open this notebook from the fraud-detection-system project."
    )

loaded_artifact = joblib.load(MODEL_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

# Some training notebooks save a dictionary containing the actual estimator.
# Unwrap common keys; otherwise stop with a useful diagnostic rather than failing later.
model = loaded_artifact
if isinstance(loaded_artifact, dict):
    print("Saved artifact is a dictionary. Available keys:", list(loaded_artifact.keys()))
    candidate_keys = [
        "model", "pipeline", "best_model", "best_estimator",
        "estimator", "classifier", "fraud_detection_pipeline"
    ]
    model = next(
        (loaded_artifact[key] for key in candidate_keys
         if key in loaded_artifact and hasattr(loaded_artifact[key], "predict_proba")),
        None
    )
    if model is None:
        # If exactly one value is a probability-capable estimator, use it.
        candidates = [
            value for value in loaded_artifact.values()
            if hasattr(value, "predict_proba")
        ]
        if len(candidates) == 1:
            model = candidates[0]
        else:
            raise TypeError(
                "The saved pickle is a dictionary, but no unique predict_proba estimator was found. "
                "Inspect the printed keys and update Notebook 06 to save the trained pipeline directly."
            )

if not hasattr(model, "predict_proba"):
    raise TypeError(
        f"Loaded model object is {type(model).__name__}, not an estimator with predict_proba(). "
        "Check the artifact saved by Notebook 06."
    )

feature_names = metadata.get("features") or metadata.get("feature_names")
if not feature_names and isinstance(loaded_artifact, dict):
    feature_names = loaded_artifact.get("features") or loaded_artifact.get("feature_names")

if not feature_names:
    feature_names = [
        "type", "amount", "oldbalanceOrg", "oldbalanceDest", "hour_of_day",
        "night_flag", "amount_to_origin_balance_ratio", "origin_balance_was_zero",
        "origin_txn_count_prior_steps"
    ]

print("Loaded estimator type:", type(model).__name__)
print("Expected model features:", feature_names)
print("Feature count:", len(feature_names))


Project root: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
Model exists: True
Metadata exists: True
Saved artifact is a dictionary. Available keys: ['pipeline', 'threshold', 'feature_columns', 'target_column', 'time_column', 'training_cutoff_step', 'outer_training_cutoff_step', 'model_name']
Loaded estimator type: Pipeline
Expected model features: ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'hour_of_day', 'night_flag', 'amount_to_origin_balance_ratio', 'origin_balance_was_zero', 'origin_txn_count_prior_steps']
Feature count: 9


In [3]:
# Cell 3 — Define the FastAPI request schema and endpoints
from typing import Literal
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, ConfigDict

app = FastAPI(
    title="Fraud Detection API",
    description="Local API wrapping the saved Phase 1 fraud-detection pipeline.",
    version="1.0.0",
)

class TransactionFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")

    type: Literal["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT", "CASH_IN"]
    amount: float = Field(ge=0)
    oldbalanceOrg: float = Field(ge=0)
    oldbalanceDest: float = Field(ge=0)
    hour_of_day: int = Field(ge=0, le=23)
    night_flag: int = Field(ge=0, le=1)
    amount_to_origin_balance_ratio: float = Field(ge=0)
    origin_balance_was_zero: int = Field(ge=0, le=1)
    origin_txn_count_prior_steps: int = Field(ge=0)

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.get("/model-info")
def model_info():
    return {
        "model_type": type(model).__name__,
        "feature_count": len(feature_names),
        "features": feature_names,
        "threshold": metadata.get("threshold"),
    }

@app.post("/predict")
def predict(transaction: TransactionFeatures):
    # Build a single-row DataFrame in the exact feature order used for training.
    row = transaction.model_dump()
    missing = [name for name in feature_names if name not in row]
    if missing:
        raise HTTPException(status_code=422, detail=f"Missing required model features: {missing}")

    X = pd.DataFrame([{name: row[name] for name in feature_names}], columns=feature_names)

    try:
        probabilities = model.predict_proba(X)
        classes = list(model.classes_)
        fraud_index = classes.index(1)
        fraud_probability = float(probabilities[0][fraud_index])
        threshold = float(metadata.get("threshold", 0.5))
        prediction = int(fraud_probability >= threshold)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {type(exc).__name__}: {exc}")

    return {
        "is_fraud": prediction,
        "fraud_probability": fraud_probability,
        "threshold": threshold,
        "decision": "review_as_potential_fraud" if prediction else "not_flagged_by_model",
        "disclaimer": "Model output is a risk signal, not proof of fraud.",
    }

print("API endpoints defined: GET /health, GET /model-info, POST /predict")


API endpoints defined: GET /health, GET /model-info, POST /predict


In [4]:
# Cell 4 — Run API tests inside the notebook without starting a server
from fastapi.testclient import TestClient

client = TestClient(app)

health_response = client.get("/health")
assert health_response.status_code == 200, health_response.text
assert health_response.json()["status"] == "ok"
assert health_response.json()["model_loaded"] is True

info_response = client.get("/model-info")
assert info_response.status_code == 200, info_response.text
assert info_response.json()["feature_count"] == len(feature_names)

print("Health endpoint:", health_response.status_code, health_response.json())
print("Model info endpoint:", info_response.status_code)


Health endpoint: 200 {'status': 'ok', 'model_loaded': True}
Model info endpoint: 200


c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [5]:
# Cell 5 — Send a sample transaction to /predict
# Synthetic example only. This is not a real customer's transaction.
sample_transaction = {
    "type": "TRANSFER",
    "amount": 1500.0,
    "oldbalanceOrg": 2000.0,
    "oldbalanceDest": 500.0,
    "hour_of_day": 14,
    "night_flag": 0,
    "amount_to_origin_balance_ratio": 1500.0 / 2000.0,
    "origin_balance_was_zero": 0,
    "origin_txn_count_prior_steps": 3,
}

prediction_response = client.post("/predict", json=sample_transaction)
print("Prediction status:", prediction_response.status_code)
print("Prediction response:", prediction_response.json())
assert prediction_response.status_code == 200, prediction_response.text
assert "is_fraud" in prediction_response.json()
assert 0 <= prediction_response.json()["fraud_probability"] <= 1


Prediction status: 200
Prediction response: {'is_fraud': 0, 'fraud_probability': 0.3805774008605504, 'threshold': 0.7999999999999999, 'decision': 'not_flagged_by_model', 'disclaimer': 'Model output is a risk signal, not proof of fraud.'}


In [6]:
# Cell 6 — Validate request handling for invalid input
invalid_response = client.post(
    "/predict",
    json={**sample_transaction, "amount": -10}
)
assert invalid_response.status_code == 422, invalid_response.text

extra_field_response = client.post(
    "/predict",
    json={**sample_transaction, "unexpected_field": "not allowed"}
)
assert extra_field_response.status_code == 422, extra_field_response.text

print("Negative amount rejected:", invalid_response.status_code == 422)
print("Unexpected field rejected:", extra_field_response.status_code == 422)


Negative amount rejected: True
Unexpected field rejected: True


In [7]:
# Cell 7 — Save an API test report for project documentation
report_dir = PROJECT_ROOT / "notebooks" / "outputs"
report_dir.mkdir(parents=True, exist_ok=True)

test_report = pd.DataFrame([
    {"test": "GET /health", "passed": health_response.status_code == 200, "status_code": health_response.status_code},
    {"test": "GET /model-info", "passed": info_response.status_code == 200, "status_code": info_response.status_code},
    {"test": "POST /predict valid request", "passed": prediction_response.status_code == 200, "status_code": prediction_response.status_code},
    {"test": "POST /predict rejects negative amount", "passed": invalid_response.status_code == 422, "status_code": invalid_response.status_code},
    {"test": "POST /predict rejects extra field", "passed": extra_field_response.status_code == 422, "status_code": extra_field_response.status_code},
])
report_path = report_dir / "phase2_api_test_report.csv"
test_report.to_csv(report_path, index=False)

display(test_report)
print("All API tests passed:", bool(test_report["passed"].all()))
print("Report saved:", report_path)


,test,passed,status_code
0,GET /health,True,200
1,GET /model-info,True,200
2,POST /predict valid request,True,200
3,POST /predict rejects negative amount,True,422
4,POST /predict rejects extra field,True,422


All API tests passed: True
Report saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\phase2_api_test_report.csv
